# 직접 질문하기 · 실행 과정 / SQL / 근거 확인

이 노트북은 **이 worktree의 실제 금융상품 Agent 파이프라인**을 실행합니다. 로컬 HTTP 서버(18080 포트)를 거치지 않습니다. 선택한 환경의 RDB·Graph·Vector 연결 설정을 사용하며, 이 파일을 만든 것만으로 연결 성공을 보장하지는 않습니다.

1. VS Code에서 이 파일을 열고 우측 상단 **커널 선택 → Python 환경**에서 아래 Python 3.13 환경을 선택하세요.
2. **1. 환경 확인**, **2. 질문 입력** 셀을 실행하세요. 질문을 적고 실행하려면 `RUN_LIVE = True`로 바꿉니다.
3. **3. 실제 실행** 셀을 한 번 실행하세요. 아래 결과 셀은 여러 번 실행해도 API를 호출하지 않습니다.

권장 커널 경로: `C:\Users\admin\.venvs\mirae-agent\Scripts\python.exe`

처음 배포된 상태는 실제 실행이 꺼져 있어 **Run All을 눌러도 API를 호출하지 않습니다.** 실행 허용 값을 True로 저장한 뒤 Run All을 누르면 호출되므로, 공유할 때는 False로 되돌리고 출력을 지우세요. 키는 기존 .env에서 읽습니다. 노트북에 키를 붙여넣지 마세요.

질문 한 건은 독립 실행이며 이전 질문을 기억하지 않습니다. 아래의 '중간 흐름'은 코드가 관찰한 의도 JSON·계획·쿼리·데이터·노드 입출력입니다. 모델의 비공개 내부 사고 과정은 수집하지 않습니다.


## 1. 환경 확인 — API 호출 없음

Python 3.13을 그대로 사용합니다. 아래 패키지 확인은 설치 여부만 검사하며 DB 연결이나 모델 호출은 하지 않습니다. 누락된 패키지가 보이면 먼저 위 커널을 선택하세요. 다른 Python에 임의로 설치할 필요는 없습니다.

현재 .env 우선 경로는 이전 검증에 사용한 T-116 worktree이며, 없으면 이 저장소의 .env입니다. 다른 설정을 쓰려면 다음 셀의 `ENV_FILE`을 바꾸세요. 환경변수가 이미 설정돼 있으면 그 값이 우선합니다. 환경 파일 또는 소스를 바꿨다면 커널을 재시작하세요.


In [ ]:
from pathlib import Path
import sys
import os
from html import escape
from IPython.display import HTML, display, Code

# 노트북 폴더 / 저장소 / 상위 작업공간에서 시작한 커널 모두 지원합니다.
candidates = list(dict.fromkeys([Path.cwd(), *Path.cwd().parents]))
ROOT = next(
    (candidate for base in candidates
     for candidate in (base, base / "worktrees" / "T-139-catalog-sql")
     if (candidate / "test/catalog-sql/manual_trace.py").is_file()),
    None,
)
if ROOT is None:
    raise FileNotFoundError("manual_trace.py가 있는 저장소에서 노트북을 여세요. 노트북만 단독 복사하지 마세요.")
ROOT = ROOT.resolve()
os.chdir(ROOT)
helper_path = str(ROOT / "test/catalog-sql")
if helper_path not in sys.path:
    sys.path.insert(0, helper_path)
import manual_trace as trace

if trace.ROOT != ROOT:
    raise RuntimeError("다른 저장소의 manual_trace가 로드돼 있습니다. 커널을 재시작하세요.")

previous_env = ROOT.parent / "T-116-fix-sql-truth" / ".env"
ENV_FILE = previous_env if previous_env.is_file() else ROOT / ".env"
REPORT = globals().get("REPORT")  # 결과 확인을 위해 설정 셀을 다시 실행해도 기록 유지

print("Python:", sys.version.split()[0])
print("커널 실행 파일:", sys.executable)
print("실행 소스:", ROOT)
print("설정 파일:", ENV_FILE, "| 존재:", ENV_FILE.is_file())
print("패키지 설치 여부:", trace.dependency_status())
print("위 결과는 DB/API 연결 상태가 아닙니다. 아직 질문을 실행하지 않았습니다.")


## 2. 질문 입력

`QUESTION`만 바꾸면 다른 질문을 시험할 수 있습니다. 질문 목록을 자동 반복 실행하지 않습니다.

- `RUN_LIVE = True`: 다음 실행 셀에서 실제 API 호출을 허용합니다. 실행 셀에 진입하면 False로 소비됩니다.
- `ALLOW_REPEAT = False`: 같은 커널에서 같은 소스·같은 질문을 실수로 다시 실행하지 못하게 합니다. 실패한 질문도 포함합니다. 의도적으로 재시도할 때만 True로 바꾸세요.
- `ROW_LIMIT = 25`: 화면의 행 미리보기 수입니다. `None`이면 **반환된 모든 행**을 표시합니다. DB 검색 자체의 LIMIT나 데이터 범위를 늘리지는 않습니다.

전체 파이프라인을 한 번 실행하지만 의도 분석·검수·답변 생성 등 여러 모델 호출이 있을 수 있습니다. 이 노트북 실행 동안 채팅 SDK 자동 재시도와 SQL 429 대기 반복은 끄고, SQL 시도 예산은 1로 제한합니다. 모든 종류의 HTTP 요청·임베딩 내부 재시도가 한 번이라는 뜻은 아닙니다. 요금·쿼터가 소모될 수 있습니다.


In [ ]:
QUESTION = """BND의 운용사, 총보수, 투자대상을 알려줘. 확인되지 않는 항목은 확인 불가라고 표시하고 출처를 밝혀줘."""
QUESTION_ID = "manual-001"

RUN_LIVE = False
ALLOW_REPEAT = False
ROW_LIMIT = 25  # None: 반환된 모든 행 표시


## 3. 실제 실행 — 이 셀만 새로운 질문을 실행

단계가 끝날 때마다 노드 이름과 경과 시간이 표시됩니다. 자세한 로그는 세션별 `run.log`에 남습니다. 병렬 노드는 완료 순서로 표시될 수 있으며, 같은 검색 노드가 의존관계 때문에 여러 차례 등장할 수 있습니다.

오류·쿼터 제한·커널의 일반 Interrupt가 발생하면 수집된 부분 결과를 가능한 한 보존합니다. 커널 강제 종료나 프로세스 종료 시에는 trace.json이 완성되지 않을 수 있지만, 이미 기록된 events.jsonl / run.log는 남습니다. HTTP 요청 중에는 라이브러리의 타임아웃까지 중단이 지연될 수 있습니다.

`completed`는 최종 답변이 생성됐다는 뜻이지 정답 판정이 아닙니다.


In [ ]:
if not RUN_LIVE:
    print("실제 실행 꺼짐. 질문 입력 셀에서 RUN_LIVE=True로 바꾸고 그 셀을 실행한 뒤 이 셀을 실행하세요.")
else:
    RUN_LIVE = False  # 실행 버튼을 연속으로 눌러도 자동 재호출하지 않도록 소비
    REPORT = None    # 이번 실행 실패 시 이전 질문의 답변을 새 답변처럼 표시하지 않음

    def on_progress(event):
        display(HTML(
            f"<div>✓ {escape(str(event['node']))} 완료 "
            f"· {event['received_s']:.2f}초</div>"
        ))

    try:
        REPORT = trace.run_question(
            QUESTION, env_file=ENV_FILE, question_id=QUESTION_ID,
            allow_repeat=ALLOW_REPEAT, on_event=on_progress,
        )
        trace.show_summary(REPORT)
        print("저장 위치:", REPORT["run_dir"])
    except Exception as exc:
        # 환경 설정/중복 실행 차단 등 파이프라인 시작 전 실패
        print(trace.Redactor().text(f"{type(exc).__name__}: {exc}"))
        print("이전 실행 기록은 삭제되지 않습니다. 아래 저장 기록 불러오기 셀을 이용하세요.")


## 4. 저장 기록 불러오기 — API 재호출 없음 (선택)

새 실행 직후에는 이 셀을 건너뛰세요. 이전 실행을 보고 싶으면 경로를 입력합니다. `LOAD_RUN`이 비어 있으면 현재 결과를 그대로 유지합니다. 경로를 넣은 채로 Run All을 실행하면 앞에서 실행한 결과 대신 이 기록이 표시됩니다.

지원: 이번 노트북의 실행 폴더 또는 trace.json, 이전 테스트의 traces.jsonl. 과거 기록에 저장되지 않았던 입력·호출 데이터는 복구하지 않습니다.


In [ ]:
LOAD_RUN = ""
# 예: r"artifacts/runs/manual-YYYYMMDD-HHMMSS-xxxxxxxx/codex-t139-sql-0905"
# 이전 Q7 기록 예:
# LOAD_RUN = r"artifacts/runs/20260905-q7-q35-remediation/codex-t139-sql-0905/live/Q7"
SAVED_RECORD_INDEX = 0  # traces.jsonl 안에서 읽을 행. 0 = 첫 번째

if LOAD_RUN.strip():
    REPORT = trace.load_report(LOAD_RUN, index=SAVED_RECORD_INDEX)
    print("불러옴:", REPORT["loaded_from"], "| 질문:", REPORT.get("question"))
else:
    print("저장 기록을 새로 불러오지 않았습니다. 현재 REPORT 유지.")


## 5. 실행 요약 · 단계별 소요 시간 · 오류

질문, 상태, 코드 커밋/해시, 데이터 release_id(확인된 경우), 노드별 시간, 각 검색 단계의 상태를 먼저 확인하세요. 총 소요 시간은 병렬 노드 시간의 합과 다를 수 있습니다.


In [ ]:
trace.show_summary(REPORT)


## 6. 최초 의도 분석 → 검수 후 의도

상품·도메인·지표·필터·관계·기간·정렬 기준을 비교하세요. SQL이 정상이어도 여기서 BND를 잘못 해석하거나 요구 지표를 빠뜨리면 결과가 틀릴 수 있습니다. 변경이 있었다면 diff에 표시됩니다. 검수 노드를 통과하지 못한 기록은 검수본이 없습니다.


In [ ]:
import difflib

initial = (REPORT or {}).get("initial_intent")
verified = (REPORT or {}).get("verified_intent")
trace.show_json(initial, "최초 의도 분석")
trace.show_json(verified, "검수 후 의도")
if initial is not None and verified is not None:
    changes = "\n".join(difflib.unified_diff(
        trace.json_text(initial).splitlines(), trace.json_text(verified).splitlines(),
        fromfile="initial_intent", tofile="verified_intent", lineterm="",
    ))
    display(Code(changes or "(변경 없음)", language="diff"))


## 7. 실행 계획 · 엔진 라우팅 · 단계 의존관계

이 표는 **계획**입니다. 실제 실행 여부는 뒤의 노드/호출 기록 및 단계 상태와 함께 확인하세요. depends_on이 있는 단계는 앞 단계의 상품 ID나 관계 결과를 받은 뒤 실행됩니다.


In [ ]:
plan = (REPORT or {}).get("plan") or []
trace.show_table(plan, limit=None)
trace.show_json((REPORT or {}).get("route"), "RDB / Graph / Vector 라우팅")
trace.show_json(plan, "계획 전체 JSON", collapsed=True)


## 8. 노드별 실제 입력 · 출력 전체

노드 목록의 index를 `NODE_INDEX`에 넣고 이 셀만 다시 실행하세요. -1은 마지막 노드입니다. 접힌 영역을 펼치면 해당 노드가 받은 state와 반환한 업데이트를 모두 볼 수 있습니다. state 안의 trace는 코드가 남긴 처리 설명 로그입니다.

과거 형식처럼 node 입력이 없는 경우에는 저장된 출력 업데이트만 표시합니다.


In [ ]:
NODE_INDEX = -1

node_records = (REPORT or {}).get("node_runs") or (REPORT or {}).get("events") or []
trace.show_table([
    {"index": i, "node": n.get("node") or n.get("name"),
     "status": n.get("status"), "duration_s": n.get("duration_s"),
     "received_s": n.get("received_s")}
    for i, n in enumerate(node_records)
], limit=None)
if node_records and -len(node_records) <= NODE_INDEX < len(node_records):
    trace.show_event(REPORT, NODE_INDEX)
else:
    print("아직 노드 기록이 없거나 NODE_INDEX가 범위를 벗어났습니다.")


## 9. RDB — 생성 SQL · 반환 컬럼/값 · NULL · 차단 사유

총보수, AUM, 매수가능 수량 등 필요한 값이 실제 행에 있는지 확인합니다. **null은 0이 아닙니다.** 이 셀은 단계 결과에 남은 SQL을 보여줍니다. 생성됐지만 실행 전에 차단된 쿼리도 있을 수 있으므로 실제 시도와 오류는 12번 호출 기록을 확인하세요.

통합 순위 뒤 상세 정보를 추가 조회한 hydration SQL도 함께 표시됩니다.


In [ ]:
trace.show_steps(REPORT, "rdb", row_limit=ROW_LIMIT)


## 10. Graph — SPARQL · 관계/분류 근거

원본 값과 온톨로지 분류는 같지 않을 수 있습니다. 어떤 관계를 조회했고 어떤 근거가 반환됐는지 확인하세요. 결과가 비었다는 사실만으로 서버 연결 실패라고 판단하면 안 됩니다. 검색 미실행·0건·차단·연결 오류를 상태와 호출 기록에서 구분하세요.


In [ ]:
trace.show_steps(REPORT, "graph", row_limit=ROW_LIMIT)


## 11. Vector — 검색 조건 · 문서 출처 · 청크 본문

단계 메타데이터에서 검색 문장·상품 범위·필터·유사도·날짜를 확인하고, 문서별 영역에서 저장된 출처와 청크를 확인하세요. 청크는 검색된 일부 본문이지 원본 문서 전체가 아닙니다. 저장된 메타데이터에 없는 페이지나 날짜를 추정해서 채우지 않습니다.


In [ ]:
trace.show_steps(REPORT, "vector", row_limit=ROW_LIMIT)


## 12. 실제 하위 호출 전체 — SQL / SPARQL / 바인딩 / 반환 데이터

스키마 확인 쿼리, 실제 검색, 임베딩 입력, Vector 내부 SQL을 **원래 함수를 호출한 시점**에 관찰한 기록입니다. 최종 SQL 하나뿐 아니라 실패한 시도도 기록합니다. 파라미터와 전체 반환 결과는 접힌 영역에 들어 있습니다.

- SQL API는 함수에 전달된 논리 SQL을 기록합니다. 전송 직전 escape 처리 등 HTTP wire payload 자체는 아닙니다.
- Vector 경유 API의 바깥 wrapper와 내부 run_sql이 함께 기록될 수 있습니다. `transport`에 nested가 표시된 항목은 **호출 계층이지 별도 DB 요청 건수로 합산할 항목이 아닙니다.**
- 함수 호출 횟수는 모델 과금 횟수나 실제 네트워크 재시도 횟수와 같지 않습니다.
- 임베딩 벡터가 파라미터에 들어가면 길 수 있습니다. 모든 필드를 확인할 수 있지만 기본은 접혀 있습니다.


In [ ]:
INCLUDE_METADATA_CALLS = True  # False면 setup/의도 분석 노드 호출을 화면에서 제외
trace.show_calls(REPORT, include_metadata=INCLUDE_METADATA_CALLS)


## 13. 모델 호출 · 토큰 사용량 · 오류

콜백이 관찰한 채팅 모델 호출입니다. 사용량이 빈 값이면 제공되지 않았다는 뜻이며 0토큰으로 해석하지 않습니다. 임베딩 사용량·SDK 내부 재시도·정확한 요금은 이 표만으로 산정할 수 없습니다.


In [ ]:
llm_calls = (REPORT or {}).get("llm_calls") or []
print("관찰한 채팅 모델 호출:", len(llm_calls))
trace.show_table(llm_calls, limit=None)
trace.show_json(llm_calls, "모델 호출 기록 전체", collapsed=True)


## 14. 최종 답변 · 답변 근거 · 합쳐진 검색 결과

실제 질문과 최종 답변을 먼저 표시합니다. retrieved_context는 답변에 포함된 근거, merged_rows는 검색 결과를 합친 데이터입니다. think_trace는 애플리케이션이 답변에 포함한 설명 필드이며 모델의 비공개 내부 추론 로그가 아닙니다.


In [ ]:
print("질문:", (REPORT or {}).get("question") or "(아직 실행하지 않음)")
answer_object = (REPORT or {}).get("answer") or {}
if isinstance(answer_object, str):
    answer_object = {"answer": answer_object}
print("\n최종 답변:\n")
print(answer_object.get("answer") or "(최종 답변 없음 — 실행 요약의 오류와 부분 결과를 확인하세요.)")
trace.show_json(answer_object.get("retrieved_context"), "답변에 포함된 출처·근거", collapsed=True)
trace.show_json(answer_object.get("think_trace"), "앱이 제공한 처리 설명", collapsed=True)
trace.show_json((REPORT or {}).get("final_state", {}).get("merged_rows"), "합쳐진 검색 데이터 전체", collapsed=True)
trace.show_json(answer_object, "최종 응답 JSON 전체", collapsed=True)


## 15. 전체 실행 기록 · 콘솔 로그 · 파일 위치

각 실행은 독립 폴더에 저장됩니다. 결과 확인 셀에서 파일을 읽어도 API를 재호출하지 않습니다.

| 파일 | 내용 |
|---|---|
| trace.json | 질문·최종 답변·의도·계획·단계별 데이터·노드 입출력·호출 기록 |
| events.jsonl | 실행 중 수신한 노드 업데이트를 순서대로 저장 |
| calls.json / queries/*.sql, *.sparql | 실제 관찰한 호출 / 쿼리 별도 파일 |
| run.log | 실행 중 출력·오류 로그 |
| manifest.json | 코드 버전·Python·실행 시간·데이터 release_id(확인된 경우) |
| schema_snapshot.json | 해당 실행에서 조회한 스키마 스냅샷(생성된 경우) |
| answer.md | 사람이 읽는 최종 답변 |

설정 키는 수집하지 않고 알려진 비밀값은 마스킹합니다. 그러나 질문·응답·DB/문서 내용 자체는 저장되므로 기록을 외부 공유하기 전에 민감 정보를 검토하세요. 노트북 출력도 저장될 수 있습니다.


In [ ]:
SHOW_LOG = False  # True면 저장된 콘솔 로그를 읽어 표시 (API 호출 없음)

trace.show_json((REPORT or {}).get("final_state", {}).get("trace"), "코드가 남긴 처리 로그", collapsed=True)
trace.show_json(REPORT, "전체 기록 JSON — 펼쳐서 모든 필드 확인", collapsed=True)

saved_path = (REPORT or {}).get("run_dir")
loaded_path = (REPORT or {}).get("loaded_from")
run_dir = Path(saved_path) if saved_path else (Path(loaded_path).parent if loaded_path else None)
if run_dir and run_dir.is_dir():
    print("실행 기록 폴더:", run_dir)
    print("파일:", ", ".join(p.name for p in run_dir.iterdir()))
    if SHOW_LOG:
        log_path = run_dir / "run.log"
        if not log_path.is_file():
            log_path = run_dir / "pipeline.log"  # 과거 테스트 형식
        if log_path.is_file():
            display(Code(trace.Redactor().text(log_path.read_text(encoding="utf-8", errors="replace")), language="text"))
        else:
            print("저장된 콘솔 로그가 없습니다.")
else:
    print("현재 읽을 실행 폴더가 없습니다.")


## 결과를 검토할 때

1. **의도**: 상품 식별자·요구 지표·비교/순위·기간을 맞게 이해했나?
2. **계획**: 요청한 항목마다 조회 단계가 있나? 필요한 Graph/Vector 단계가 빠지지 않았나?
3. **쿼리**: 실제 실행했나? 올바른 테이블·컬럼·단위·정렬·NULL 처리를 사용했나?
4. **데이터**: 값이 없나(NULL), 행이 없나(0건), 범위가 제한됐나, 연결/권한/스키마 오류인가?
5. **답변**: 반환 값이 정확히 반영됐나? 모르는 항목을 누락하거나 값을 지어내지 않았나? 문서 출처·원본값·분류 근거·갱신일을 구분했나?

다른 질문: 2번 셀에서 QUESTION / QUESTION_ID를 바꾸고 RUN_LIVE=True → 3번 셀 실행 → 필요한 결과 셀 실행. 기존 결과를 다시 검토: 4번 셀에서 저장 폴더 불러오기 → 5번 이후 셀 실행.
